In [31]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
!mkdir -p /content/drive/MyDrive/aiffel/word_embedding

In [33]:
import os
os.chdir('/content/drive/MyDrive/aiffel/word_embedding')

In [34]:
# !sudo apt update
# !sudo apt install openjdk-17-jdk -y
# !echo 'export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))' >> ~/.bashrc
# !source ~/.bashrc
# 위 명령어는 로컬 작업 시 참고용입니다.
# Colab 에는 OpenJDK 가 이미 설치되어 있어 KoNLPy(Okt)가 바로 동작합니다.

!pip install konlpy

In [35]:
import re
from konlpy.tag import Okt
from collections import Counter
print("임포트 완료")

임포트 완료


In [36]:
text = "임금님 귀는 당나귀 귀! 임금님 귀는 당나귀 귀! 실컷~ 소리치고 나니 속이 확 뚫려 살 것 같았어."
text

'임금님 귀는 당나귀 귀! 임금님 귀는 당나귀 귀! 실컷~ 소리치고 나니 속이 확 뚫려 살 것 같았어.'

In [37]:
reg = re.compile("[^ㄱ-ㅎㅏ-ㅣ가-힣 ]")
text = reg.sub('', text)
print(text)

임금님 귀는 당나귀 귀 임금님 귀는 당나귀 귀 실컷 소리치고 나니 속이 확 뚫려 살 것 같았어


In [38]:
%%time
okt=Okt()                          # Okt 형태소 분석기 객체를 생성
tokens = okt.morphs(text)          # 문장을 형태소 단위로 쪼개어 리스트 형태로 반환
print(tokens)

['임금님', '귀', '는', '당나귀', '귀', '임금님', '귀', '는', '당나귀', '귀', '실컷', '소리', '치고', '나니', '속이', '확', '뚫려', '살', '것', '같았어']
CPU times: user 67.7 ms, sys: 0 ns, total: 67.7 ms
Wall time: 59.6 ms


In [39]:
vocab = Counter(tokens)
print(vocab)

Counter({'귀': 4, '임금님': 2, '는': 2, '당나귀': 2, '실컷': 1, '소리': 1, '치고': 1, '나니': 1, '속이': 1, '확': 1, '뚫려': 1, '살': 1, '것': 1, '같았어': 1})


In [40]:
vocab['임금님'], vocab['귀']

(2, 4)

In [41]:
vocab_size = 5
vocab = vocab.most_common(vocab_size) # 등장 빈도수가 높은 상위 5개의 단어만 저장
print(vocab)

[('귀', 4), ('임금님', 2), ('는', 2), ('당나귀', 2), ('실컷', 1)]


In [42]:
word2idx={word[0] : index+1 for index, word in enumerate(vocab)}
print(word2idx)

{'귀': 1, '임금님': 2, '는': 3, '당나귀': 4, '실컷': 5}


In [43]:
def one_hot_encoding(word, word2index):
       one_hot_vector = [0]*(len(word2index))
       index = word2index[word]
       one_hot_vector[index-1] = 1
       return one_hot_vector
print("슝=3")

슝=3


In [44]:
one_hot_encoding("임금님", word2idx)

[0, 1, 0, 0, 0]

In [45]:
import pandas as pd

pd.Series(word2idx).head()

,0
귀,1
임금님,2
는,3
당나귀,4
실컷,5


In [46]:
import torch
import torch.nn.functional as F
from collections import Counter
print("임포트 완료")

임포트 완료


In [47]:
text = [['강아지', '고양이', '강아지'],['애교', '고양이'], ['컴퓨터', '노트북']]
text

[['강아지', '고양이', '강아지'], ['애교', '고양이'], ['컴퓨터', '노트북']]

In [48]:
text = [['강아지', '고양이', '강아지'], ['애교', '고양이'], ['컴퓨터', '노트북']] # 데이터
counter = Counter(word for sentence in text for word in sentence) # 단어 등장 빈도 계산
word_index = {word: i+1 for i, (word, _) in enumerate(counter.most_common())}
word_index["<PAD>"] = 0   # OOV(Out-of-Vocabulary) 처리를 위해 기본 인덱스 설정

print(word_index)

{'강아지': 1, '고양이': 2, '애교': 3, '컴퓨터': 4, '노트북': 5, '<PAD>': 0}


In [49]:
vocab_size = len(word_index) # 단어 사전 크기
print("슝=3")

슝=3


In [50]:
sub_text = ['강아지', '고양이', '강아지', '컴퓨터']
encoded = [word_index.get(word, 0) for word in sub_text]  # OOV 단어는 0으로 처리
print(encoded)

[1, 2, 1, 4]


In [51]:
encoded_tensor = torch.tensor(encoded, dtype=torch.long) # PyTorch 텐서 변환
one_hot = F.one_hot(encoded_tensor, num_classes=len(word_index))
print(one_hot)

tensor([[0, 1, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 1, 0, 0, 0, 0],
        [0, 0, 0, 0, 1, 0]])


In [52]:
!pip install -q gensim nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 66.0 MB/s eta 0:00:00


In [53]:
import nltk
nltk.download('abc')
nltk.download('punkt') # 구버전 호환을 위해 함께 다운로드
nltk.download('punkt_tab')

[nltk_data] Downloading package abc to /root/nltk_data...
[nltk_data]   Unzipping corpora/abc.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [54]:
from nltk.corpus import abc
corpus = abc.sents()
print("슝~")

슝~


In [55]:
print(corpus[:3])

[['PM', 'denies', 'knowledge', 'of', 'AWB', 'kickbacks', 'The', 'Prime', 'Minister', 'has', 'denied', 'he', 'knew', 'AWB', 'was', 'paying', 'kickbacks', 'to', 'Iraq', 'despite', 'writing', 'to', 'the', 'wheat', 'exporter', 'asking', 'to', 'be', 'kept', 'fully', 'informed', 'on', 'Iraq', 'wheat', 'sales', '.'], ['Letters', 'from', 'John', 'Howard', 'and', 'Deputy', 'Prime', 'Minister', 'Mark', 'Vaile', 'to', 'AWB', 'have', 'been', 'released', 'by', 'the', 'Cole', 'inquiry', 'into', 'the', 'oil', 'for', 'food', 'program', '.'], ['In', 'one', 'of', 'the', 'letters', 'Mr', 'Howard', 'asks', 'AWB', 'managing', 'director', 'Andrew', 'Lindberg', 'to', 'remain', 'in', 'close', 'contact', 'with', 'the', 'Government', 'on', 'Iraq', 'wheat', 'sales', '.']]


In [56]:
print('코퍼스의 크기 :',len(corpus))

코퍼스의 크기 : 29059


In [57]:
%%time
from gensim.models import Word2Vec

model = Word2Vec(sentences = corpus, vector_size = 100, window = 5, min_count = 5, workers = 4, sg = 0)
print("모델 학습 완료!")

모델 학습 완료!
CPU times: user 18.5 s, sys: 114 ms, total: 18.7 s
Wall time: 14.2 s


In [58]:
model_result = model.wv.most_similar("man")
print(model_result)

[('woman', 0.9236652255058289), ('skull', 0.9211822152137756), ('asteroid', 0.902588427066803), ('Bang', 0.9018574357032776), ('third', 0.8985815644264221), ('mutation', 0.8977639079093933), ('dog', 0.8953710794448853), ('dinosaur', 0.8939290046691895), ('rally', 0.8937207460403442), ('symbol', 0.8906550407409668)]


In [61]:
from gensim.models import KeyedVectors

# 경로는 본인 설정에 따라 필요하면 수정하세요.
model.wv.save_word2vec_format('w2v')
loaded_model = KeyedVectors.load_word2vec_format('w2v')
print("모델  load 완료!")

모델  load 완료!


In [62]:
model_result = loaded_model.most_similar("man")
print(model_result)

[('woman', 0.9236652255058289), ('skull', 0.9211822152137756), ('asteroid', 0.902588427066803), ('Bang', 0.9018574357032776), ('third', 0.8985815644264221), ('mutation', 0.8977639079093933), ('dog', 0.8953710794448853), ('dinosaur', 0.8939290046691895), ('rally', 0.8937207460403442), ('symbol', 0.8906550407409668)]


In [63]:
# 에러가 나더라도 놀라지 마세요.
loaded_model.most_similar('overacting')

KeyError: "Key 'overacting' not present in vocabulary"

In [64]:
# 에러가 나더라도 놀라지 마세요.
loaded_model.most_similar('memorry')

KeyError: "Key 'memorry' not present in vocabulary"

In [65]:
# 경로 주의
!python -m gensim.scripts.word2vec2tensor --input w2v --output w2v

2026-09-03 07:24:35,686 - word2vec2tensor - INFO - running /usr/local/lib/python3.13/dist-packages/gensim/scripts/word2vec2tensor.py --input w2v --output w2v
2026-09-03 07:24:35,686 - keyedvectors - INFO - loading projection weights from w2v
2026-09-03 07:24:36,928 - utils - INFO - KeyedVectors lifecycle event {'msg': 'loaded (10363, 100) matrix of type float32 from w2v', 'binary': False, 'encoding': 'utf8', 'datetime': '2026-09-03T07:24:36.927013', 'gensim': '4.4.0', 'python': '3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]', 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.35', 'event': 'load_word2vec_format'}
2026-09-03 07:24:39,939 - word2vec2tensor - INFO - 2D tensor file saved to w2v_tensor.tsv
2026-09-03 07:24:39,940 - word2vec2tensor - INFO - Tensor metadata file saved to w2v_metadata.tsv
2026-09-03 07:24:39,941 - word2vec2tensor - INFO - finished running word2vec2tensor.py


In [73]:
!ls

w2v  w2v_metadata.tsv  w2v_tensor.tsv


In [74]:
%%time
from gensim.models import FastText
fasttext_model = FastText(corpus, window=5, min_count=5, workers=4, sg=1)
print("FastText 학습 완료!")

FastText 학습 완료!
CPU times: user 1min 8s, sys: 366 ms, total: 1min 8s
Wall time: 45.8 s


In [75]:
fasttext_model.wv.most_similar('overacting')

[('extracting', 0.9457700252532959),
 ('overwhelming', 0.9395087361335754),
 ('mixing', 0.9381648898124695),
 ('emptying', 0.9374138116836548),
 ('fluctuating', 0.9368230104446411),
 ('resolving', 0.9283294677734375),
 ('fixing', 0.9282210469245911),
 ('enjoying', 0.9281095266342163),
 ('lifting', 0.9276623725891113),
 ('hurting', 0.9267507195472717)]

In [76]:
fasttext_model.wv.most_similar('memoryy')

[('memory', 0.950620710849762),
 ('musical', 0.8754053711891174),
 ('intelligence', 0.8634200692176819),
 ('basic', 0.8631806969642639),
 ('emotion', 0.8611988425254822),
 ('anticipate', 0.8610060811042786),
 ('personal', 0.8567429780960083),
 ('mechanism', 0.8556057810783386),
 ('mechanisms', 0.8527159094810486),
 ('consequence', 0.8506069779396057)]

In [77]:
import gensim.downloader as api
glove_model = api.load("glove-wiki-gigaword-50")  # glove vectors 다운로드
glove_model.most_similar("dog")  # 'dog'과 비슷한 단어 찾기

[==================================================] 100.0% 66.0/66.0MB downloaded


[('cat', 0.9218004941940308),
 ('dogs', 0.8513158559799194),
 ('horse', 0.7907583713531494),
 ('puppy', 0.7754920721054077),
 ('pet', 0.7724708318710327),
 ('rabbit', 0.7720814347267151),
 ('pig', 0.7490062117576599),
 ('snake', 0.7399188876152039),
 ('baby', 0.7395570278167725),
 ('bite', 0.7387937307357788)]

In [78]:
glove_model.most_similar('overacting')

[('impudence', 0.7842012047767639),
 ('puerile', 0.7816032767295837),
 ('winningly', 0.7644237875938416),
 ('grossness', 0.7576098442077637),
 ('deconstructions', 0.748936653137207),
 ('over-the-top', 0.7460805773735046),
 ('buffoonery', 0.746045708656311),
 ('impetuosity', 0.7415392398834229),
 ('sophomoric', 0.736961841583252),
 ('zaniness', 0.7353197336196899)]

In [79]:
# 에러가 나더라도 놀라지 마세요.
glove_model.most_similar('memoryy')

KeyError: "Key 'memoryy' not present in vocabulary"